# NB4 — Negative V1 → scorer-ready dataset → final freeze

Notebook này hoàn tất data processing V1 sau khi NB3 đã PASS:

1. sinh **1 synthetic negative / 1 final positive**;
2. replacement cùng `master_category`, khác item, khác kit và cùng official split;
3. ghép từng positive với negative thành scorer-ready JSONL;
4. validator tự dựng lại từng cặp để kiểm tra đúng một swap và toàn bộ provenance;
5. kiểm tra label balance, metadata, embedding gate và cross-split leakage;
6. tạo SHA-256 manifests và chỉ trả `READY_TO_TRAIN` khi mọi gate đều pass.

> Negative V1 vẫn là random synthetic negative, không phải hard negative.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Lấy core code từ GitHub

Notebook mặc định dùng `main`. Khi test PR trước merge, tạm đổi `REPO_REF` thành feature branch trong Colab.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = 'https://github.com/ThinhTran2208/opisoverated.git'
REPO_REF = 'main'  # Test trước merge: tạm đổi thành feature branch.
REPO_DIR = Path('/content/opisoverated')

if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_REF], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_REF], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)], check=True)

GIT_COMMIT = subprocess.check_output(
    ['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True
).strip()
sys.path.insert(0, str(REPO_DIR))
print('Using ref   :', REPO_REF)
print('Git commit :', GIT_COMMIT)

## 3. Khai báo input/output

Input giữ nguyên trong `core7_drop_v1`. Scorer-ready artifacts được ghi sang folder riêng để không đụng vào clean positives.

In [ ]:
CORE7_DIR = Path('/content/drive/MyDrive/fashion_audit/core7_drop_v1')
OUTPUT_DIR = Path('/content/drive/MyDrive/fashion_audit/scorer_ready_v1')
EMBEDDING_REPORT = CORE7_DIR / 'core7_embedding_validation_report.json'
SEED = 42
ALLOW_OVERWRITE = False

required_paths = [EMBEDDING_REPORT]
for split in ('train', 'valid', 'test'):
    required_paths.extend([
        CORE7_DIR / f'category_clean_{split}.jsonl',
        CORE7_DIR / f'core7_item_metadata_v1_{split}.jsonl',
    ])

for path in required_paths:
    print(path.name, 'exists=', path.exists())
    if not path.exists():
        raise FileNotFoundError(path)

print('Output folder:', OUTPUT_DIR)
print('Seed         :', SEED)

## 4. Build full scorer-ready dataset V1

`ALLOW_OVERWRITE=False` bảo vệ versioned artifacts. Nếu V1 đã tồn tại, notebook dừng thay vì âm thầm tạo benchmark khác cùng tên.

In [ ]:
from src.data.build_core7_scorer_dataset import build_scorer_dataset_v1

result = build_scorer_dataset_v1(
    data_dir=CORE7_DIR,
    output_dir=OUTPUT_DIR,
    embedding_report_path=EMBEDDING_REPORT,
    seed=SEED,
    git_commit=GIT_COMMIT,
    overwrite=ALLOW_OVERWRITE,
)

## 5. Đọc negative sampling report

In [ ]:
for split in ('train', 'valid', 'test'):
    sampling = result['sampling_reports'][split]
    merge = sampling['merge']
    print(split.upper())
    print('  positive input     :', sampling['positive_count'])
    print('  negatives generated:', sampling['negative_count'])
    print('  generation coverage:', f"{sampling['generation_coverage']:.4%}")
    print('  failed positives   :', sampling['failed_positive_count'])
    print('  sampling pass      :', sampling['pass'])
    print('  merge pass         :', merge['pass'])
    if sampling['failure_examples']:
        print('  failure examples   :', sampling['failure_examples'][:5])
    print()

## 6. Đọc final validation gate

In [ ]:
final_report = result['final_validation']
for split in ('train', 'valid', 'test'):
    split_report = final_report['splits'][split]
    print(split.upper())
    print('  samples   :', split_report['sample_count'])
    print('  positives :', split_report['positive_count'])
    print('  negatives :', split_report['negative_count'])
    print('  issues    :', split_report['issue_count'])
    print('  pass      :', split_report['pass'])
    if split_report['issue_examples']:
        print('  examples  :', split_report['issue_examples'][:5])
    print()

print('Embedding gate              :', final_report['embedding_validation_pass'])
print('Negative sampling gate      :', final_report['negative_sampling_pass'])
print('Source-kit cross-split      :', final_report['source_kit_cross_split_count'])
print('Item cross-split            :', final_report['item_cross_split_count'])
print('Global duplicate sample IDs :', final_report['global_duplicate_sample_id_count'])
print('FINAL STATUS                :', result['status'])

## 7. Expected outputs

Folder `scorer_ready_v1` sẽ chứa:

```text
negative_v1_train/valid/test.jsonl
scorer_ready_v1_train/valid/test.jsonl
negative_sampling_v1_*_report.json
final_validation_v1.json
split_manifest_v1.json
dataset_manifest_v1.json
```

Chỉ khi `FINAL STATUS = READY_TO_TRAIN` thì Type-aware Pairwise scorer mới được dùng các `scorer_ready_v1_*` để train/validation/test.